# Task 1 - Source Extraction

## KAUST Data Extraction

KAUST research data is collected from two sources:

- 2023: KAUST repository source file
- 2024–2025: Crossref REST API using the KAUST ROR identifier

Raw source data is stored in `data/raw/` without cleaning or transformation.

In [1]:
import requests
import json
import time
from pathlib import Path
from datetime import datetime

### KAUST 2023 Repository Data

The 2023 KAUST dataset is provided as a raw repository CSV file and is retained without modification.

In [2]:
raw_dir = Path("../data/raw")

kaust_2023_file = raw_dir / "KAUST_2023_raw.csv"

print("KAUST 2023 file exists:", kaust_2023_file.exists())
print("File:", kaust_2023_file)

KAUST 2023 file exists: True
File: ..\data\raw\KAUST_2023_raw.csv


### KAUST 2024–2025 - Crossref API

Source: Crossref REST API

Endpoint: https://api.crossref.org/works

Institution: King Abdullah University of Science and Technology (KAUST)

KAUST ROR ID: 01q3tbs38

Publication period: 2024-01-01 to 2025-12-31

Authentication: No API key required.

The API response is saved as raw JSON without modification.

In [3]:
url = "https://api.crossref.org/works"

kaust_ror = "01q3tbs38"

date_filter = (
    f"ror-id:{kaust_ror},"
    "from-pub-date:2024-01-01,"
    "until-pub-date:2025-12-31"
)

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

print(date_filter)

ror-id:01q3tbs38,from-pub-date:2024-01-01,until-pub-date:2025-12-31


In [4]:
params = {
    "filter": date_filter,
    "rows": 5
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("URL:", response.url)

Status: 200
URL: https://api.crossref.org/works?filter=ror-id%3A01q3tbs38%2Cfrom-pub-date%3A2024-01-01%2Cuntil-pub-date%3A2025-12-31&rows=5


In [5]:
response.raise_for_status()

test_data = response.json()

total_results = test_data["message"]["total-results"]

print("Total KAUST records found:", total_results)

Total KAUST records found: 124


In [6]:
cursor = "*"
page = 1
total_records = 0

while True:
    params = {
        "filter": date_filter,
        "rows": 100,
        "cursor": cursor
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        page_data = response.json()
        items = page_data["message"]["items"]

        if not items:
            break

        output_file = raw_dir / f"KAUST_Crossref_2024_2025_page_{page}.json"

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(response.text)

        total_records += len(items)

        print(
            f"Page {page}: {len(items)} records | "
            f"Total: {total_records}"
        )

        cursor = page_data["message"].get("next-cursor")

        if not cursor:
            break

        if total_records >= page_data["message"]["total-results"]:
            break

        page += 1
        time.sleep(1.2)

    except requests.exceptions.Timeout:
        print("Request timed out.")
        break

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        break

Page 1: 100 records | Total: 100
Page 2: 24 records | Total: 124


In [7]:
print("Extraction completed.")
print("Total records downloaded:", total_records)

Extraction completed.
Total records downloaded: 124


In [8]:
list(raw_dir.glob("KAUST_Crossref_2024_2025_page_*.json"))

[WindowsPath('../data/raw/KAUST_Crossref_2024_2025_page_1.json'),
 WindowsPath('../data/raw/KAUST_Crossref_2024_2025_page_2.json')]

## KFUPM Data Extraction

KFUPM research metadata is collected programmatically from the KFUPM EPrints repository.

The extraction will retrieve research records and save the raw response in `data/raw/` without cleaning or transformation.

In [9]:
kfupm_url = "https://eprints.kfupm.edu.sa/cgi/search/simple"

params = {
    "q": "Computer Engineering",
    "output": "JSON"
}

response = requests.get(
    kfupm_url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text[:500])

Status: 200
Content-Type: application/json; charset=utf-8
[

]




In [10]:
kfupm_url = "https://eprints.kfupm.edu.sa/cgi/exportview/divisions/C1/2025/JSON/C1_2025.js"

response = requests.get(
    kfupm_url,
    timeout=30
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text[:500])

Status: 200
Content-Type: application/json; charset=utf-8
[
    {
      "contact_email": "Mdikko14@gmail.com",
      "divisions": [
        "C1"
      ],
      "co_advisor": [
        {
          "name": {
            "lineage": null,
            "given": "Basem",
            "honourific": null,
            "family": "Al-madani"
          },
          "id": "mbasem@kfupm.edu.sa"
        }
      ],
      "eprintid": 143947,
      "lastmod": "2026-06-30 09:20:06",
      "creators": [
        {
          "name": {
            "family": "Gambo",
          


### KFUPM Computer Engineering Extraction

Source: KFUPM EPrints Repository

Division: Computer Engineering (C1)

Years: 2023–2026

The pipeline retrieves the JSON export programmatically and stores each raw response without modification in `data/raw/`.

In [11]:
kfupm_years = [2023, 2024, 2025, 2026]

for year in kfupm_years:

    url = (
        f"https://eprints.kfupm.edu.sa/cgi/exportview/"
        f"divisions/C1/{year}/JSON/C1_{year}.js"
    )

    try:
        response = requests.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        output_file = raw_dir / f"KFUPM_C1_{year}_raw.json"

        with open(output_file, "wb") as f:
            f.write(response.content)

        records = response.json()

        print(
            f"{year}: {len(records)} records saved"
        )

        time.sleep(1)

    except requests.exceptions.Timeout:
        print(f"{year}: Request timed out")

    except requests.exceptions.RequestException as e:
        print(f"{year}: Request failed - {e}")

2023: 2 records saved
2024: 16 records saved
2025: 13 records saved
2026: 17 records saved


In [12]:
list(raw_dir.glob("KFUPM_C1_*_raw.json"))

[WindowsPath('../data/raw/KFUPM_C1_2023_raw.json'),
 WindowsPath('../data/raw/KFUPM_C1_2024_raw.json'),
 WindowsPath('../data/raw/KFUPM_C1_2025_raw.json'),
 WindowsPath('../data/raw/KFUPM_C1_2026_raw.json')]